# 11 Local Model Serving with FastAPI and Streamlit

So far, we can train a model, save artifacts, run inference from Python, and execute tests. The next step is local model serving.

In this notebook, we expose the Bank Marketing model through a FastAPI service and use Streamlit as a small user-facing client.

## 1. Real-Time Serving versus Batch Inference

The CLI pipeline from the previous notebook is useful for batch inference: a file goes in, and a file with predictions comes out.

An API supports real-time inference. A client sends one request, and the model service returns one response. This is useful when another system needs a prediction immediately, for example during a bank marketing campaign.

## 2. The API as a Contract

The API defines a contract between the model service and the consuming system.

In this project, the contract is intentionally small:

- `GET /health`: checks whether the service is reachable
- `GET /model-info`: returns metadata about the served model artifacts
- `POST /predict`: accepts one customer record and returns one subscription prediction

FastAPI automatically exposes the OpenAPI specification and Swagger UI:

`http://127.0.0.1:8000/docs`

`http://127.0.0.1:8000/openapi.json`

The Swagger page is useful for teaching because it makes the request and response schema visible.

## 3. Serving Artifacts

The API loads the model and preprocessor through a local manifest:

`models/model_manifest.json`

The manifest points to:

`models/<timestamp>_XGBClassifier_tuned_candidate.joblib`

`models/<timestamp>_preprocessor.joblib`

This is important because the model alone is not enough. The API must use the exact preprocessor fitted during training.

In [2]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

manifest_path = PROJECT_ROOT / "models" / "model_manifest.json"
with manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

manifest

{'model_name': 'XGBoost Default',
 'model_class': 'XGBClassifier',
 'created_at': '20260723_173943',
 'model_path': '/home/patri/master/ads2-bank-marketing-project/models/20260723_173943_XGBClassifier_model.joblib',
 'preprocessor_path': '/home/patri/master/ads2-bank-marketing-project/models/20260723_173943_preprocessor.joblib',
 'threshold': 0.5,
 'mlflow_run_id': '31fe93297cc546e7928655a0843b7c36'}

## 4. Start the FastAPI Service

Run this command from the repository root:

```bash
uvicorn src.serving.api:app --reload --port 8000
```

Then open Swagger UI:

```text
http://127.0.0.1:8000/docs
```

The server process must keep running while you use the API.

## 5. Example API Request

The `/predict` endpoint expects one customer as JSON. The target `y` is not part of the request because it is unknown at inference time. The `duration` variable is also excluded because it was removed to avoid data leakage.

In [3]:
example_customer = {
    "age": 40,
    "job": "management",
    "marital": "married",
    "education": "tertiary",
    "default": "no",
    "balance": 1500,
    "housing": "yes",
    "loan": "no",
    "contact": "cellular",
    "day": 15,
    "month": "may",
    "campaign": 2,
    "pdays": -1,
    "previous": 0,
    "poutcome": "unknown",
}

example_customer

{'age': 40,
 'job': 'management',
 'marital': 'married',
 'education': 'tertiary',
 'default': 'no',
 'balance': 1500,
 'housing': 'yes',
 'loan': 'no',
 'contact': 'cellular',
 'day': 15,
 'month': 'may',
 'campaign': 2,
 'pdays': -1,
 'previous': 0,
 'poutcome': 'unknown'}

With the API running, this request can be sent from Python:

In [4]:
# This cell requires the FastAPI server to be running.
import requests

response = requests.post("http://127.0.0.1:8000/predict", json=example_customer, timeout=10)
response.raise_for_status()
response.json()

{'subscription_probability': 0.08765621483325958,
 'subscription_prediction': 0,
 'threshold': 0.5,
 'model_name': 'XGBoost Default',
 'model_version': '20260723_173943'}

## 6. Streamlit as a Simple Client

The Streamlit app does not load the model directly. It calls the FastAPI service.

Start it from the repository root after the API is running:

`streamlit run app/streamlit_app.py`

This separation is useful: the API owns model inference, while Streamlit is only one possible user interface.


## 7. Local Demo versus Production Deployment

This setup is local. It demonstrates the serving architecture and the API contract, but it is not yet a production deployment.

A production setup would need additional concerns such as authentication, logging, rate limits, containerization, deployment configuration, monitoring, and secure artifact management.

For this course step, the important idea is the separation of responsibilities:

- the training pipeline creates artifacts
- the manifest selects the artifacts used for serving
- FastAPI exposes a stable prediction contract
- Streamlit acts as a non-technical client